In [ ]:
from fidelity_option_data_downloader import FidelityOptionDataDownloader
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')
ocd = FidelityOptionDataDownloader('chain', 'quotes', 'cookie.txt', logger=self.logger)
pd.set_option("display.max_columns", None)

In [ ]:
def plot_gex_profile(df, symbol, last_price):
    px.scatter(df[df.symbol==symbol], x='strike', y='dollar_gex', title=f'Last price: {last_price}', height=1000).show()

In [ ]:
def time_option_data_retrieval(symbol):
    t0 = time.perf_counter()
    res = ocd.get_slo_chain_data(symbol)
    print(time.perf_counter() - t0)

In [ ]:
with open('symbols.txt') as fo:
    symlist = [x.rstrip() for x in fo]
len(symlist)

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_raw = self.build_option_df(symlist)
dfcp = self.concat_put_call_options(df_raw).drop(columns=['TimeValue', 'IntrinsicValue', 'Rho', 'dte_cluster'])
px.bar(self.check_data_age(df_raw), barmode='group', height=300, title='Data Ages', width=1500).show()

In [ ]:
total_net_gex, strike_gex = self.do_gex(dfcp)
_title = f'{df_raw.__.load_dt.max()}'
px.bar(total_net_gex, x='symbol', y='dollar_gex', title=_title, height=400, width=1500).show()
total_net_gex.sort_values(by='dollar_gex')

In [ ]:
_sym = 'MSFT'
plot_gex_profile(strike_gex, _sym, dfcp[dfcp.symbol==_sym].lastPrice.iloc[0].item())

In [ ]:
dfp = dfcp[dfcp.type=='P']
dfp = self.compute_all_time_decay_metrics_for_symbols(dfp, [symbol], ignore_no_bid=True, exclude_0dte=False, oi_lb=100)
dfp['hdtePriceOvStrike'] = 100*dfp.lastPrice*(1 - dfp.ImpVola * np.sqrt(dfp.dte/730))/dfp.strike - 100
dfp.sort_values(by='dteProfit', ascending=False).head(60)

In [ ]:
px.scatter(dfp, x='strike', y='OpenInterest', color='expDt')

In [ ]:
dfp[dfp.strike==670].drop(columns=['E', 'moneyness', 'dtz'])

In [ ]:
px.scatter(dfp[(dfp.dte==0) & (dfp.mid >= 0.5)], y='mid', x='OpenInterest', color='strike')

In [ ]:
#get_roll_options(dfp, 'QQQ', 578, 0, 1.29, dte_ub=8, extra_loss=0).head(60)
px.scatter(get_roll_options(dfp, 'SPY', 650, 0, 1.31, dte_ub=5, extra_loss=0), x='strike', y='mid', color='expDt')

In [ ]:
_dfp = dfp[(dfp.hdtePriceOvStrike >= 0) & (dfp.hdteProfit >= 24)].sort_values(by='hdteProfit', ascending=False)
px.scatter(_dfp, x='Delta', y='hdteProfit', color='expDt', width=1200, height=600).show()
_dfp.head(60)

### Sell Calls

In [ ]:
dfc = dfcp[dfcp.type=='C']
dfc = self.compute_all_time_decay_metrics_for_symbols(dfc, [symbol], ignore_no_bid=True, exclude_0dte=True, oi_lb=100)
#dfc['dteStrikeMargin'] = 100*dfc.lastPrice*(1 - 3*dfc.ImpVola * np.sqrt(dfc.dte/365))/dfc.strike - 100
dfc

In [ ]:
dfc[(dfc.strike >= 145) & (dfc.mid >= 1) & (dfc.dte <= 60)].drop(columns=['dth', 'dtz', 'overpaid', 'leverage', 'dtzr']).sort_values(by='dthProfit', ascending=False)

In [ ]:
px.scatter(dfc[(dfc.dte <= 60) & (dfc.strike >= 145)], x='Delta', y='mid', color='expDt')